# LeetCode 3864 — Minimum Cost to Partition a Binary String bằng Python

- Link: https://leetcode.com/problems/minimum-cost-to-partition-a-binary-string/
- Độ khó: **Hard**
- Chủ đề: Chia để trị (đệ quy chia đôi), quy hoạch động trên cây chia đôi, tổng tiền tố, DP từ dưới lên theo tầng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCost(self, s: str, encCost: int, flatCost: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra cấu trúc **cây chia đôi**: mỗi đoạn có độ dài chẵn chỉ có **một** cách tách (ở giữa), nên tập các đoạn có thể xuất hiện là cố định và chỉ có `O(n)` đoạn.
2. Viết đệ quy `best(l, r) = min(chi phí giữ nguyên, best(l, m) + best(m, r))`.
3. Dùng **tổng tiền tố** để đếm số ký tự `'1'` trong một đoạn trong `O(1)`.
4. Chuyển đệ quy thành **DP từ dưới lên theo tầng** (không đệ quy).
5. Phân tích độ phức tạp: tổng độ dài các đoạn ở mỗi tầng là `n`, nhưng nhờ tiền tố mỗi đoạn chỉ tốn `O(1)`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho xâu nhị phân `s` (`'1'` = phần tử nhạy cảm) và hai số `encCost`, `flatCost`. Ban đầu cả xâu là một đoạn. Chi phí một đoạn độ dài `L` có `X` ký tự `'1'`:

- `X = 0`: chi phí `flatCost`;
- `X > 0`: chi phí `L · X · encCost`.

Một đoạn có **độ dài chẵn** có thể được **tách đôi** thành hai đoạn bằng nhau; khi đó chi phí của nó là tổng chi phí hai nửa (mỗi nửa lại có thể tách tiếp nếu độ dài chẵn). Tìm tổng chi phí nhỏ nhất.

**Ví dụ 1:** `s = "1010", encCost = 2, flatCost = 1` → `6` (tách hết thành 4 ký tự: `2 + 1 + 2 + 1`).

**Ví dụ 2:** `s = "1010", encCost = 3, flatCost = 10` → `12` (tách thành `"10"`, `"10"`, mỗi đoạn `2·1·3 = 6`).

**Ví dụ 3:** `s = "00", encCost = 1, flatCost = 2` → `2`.

**Ràng buộc:** `1 ≤ |s| ≤ 10^5`, `1 ≤ encCost, flatCost ≤ 10^5`.

In [1]:
import random, time, sys

examples = [("1010", 2, 1, 6), ("1010", 3, 10, 12), ("00", 1, 2, 2)]
for s, e, f, exp in examples:
    print(repr(s), e, f, "-> kỳ vọng", exp)

'1010' 2 1 -> kỳ vọng 6
'1010' 3 10 -> kỳ vọng 12
'00' 1 2 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Chi phí một đoạn

Viết thành hàm để dùng lại. `str.count('1')` đếm trong `O(L)`.

In [2]:
def seg_cost(seg, enc, flat):
    x = seg.count('1')
    return len(seg) * x * enc if x else flat

print(seg_cost("1010", 2, 1), seg_cost("10", 2, 1), seg_cost("0", 2, 1), seg_cost("000", 5, 7))

16 4 1 7


### 3.2. Tổng tiền tố đếm số `'1'`

`pre[i]` = số `'1'` trong `s[0..i-1]`. Số `'1'` trong `[l, r)` là `pre[r] - pre[l]`.

In [3]:
s = "1101001"
pre = [0]
for c in s:
    pre.append(pre[-1] + (c == '1'))
print(pre, "| số '1' trong [2, 6):", pre[6] - pre[2], "=", s[2:6].count('1'))

[0, 1, 2, 2, 3, 3, 3, 4] | số '1' trong [2, 6): 1 = 1


### 3.3. Cây chia đôi

Từ đoạn `[0, n)`: nếu độ dài chẵn thì có hai con `[l, m)` và `[m, r)`; nếu lẻ thì là lá. Mỗi tầng các đoạn rời nhau và phủ `[0, n)`. Số tầng ≤ `log₂ n + 1`; tổng số nút ≤ `2n`.

In [4]:
def halving_tree(n):
    levels = [[(0, n)]]
    while (levels[-1][0][1] - levels[-1][0][0]) % 2 == 0:
        nxt = []
        for l, r in levels[-1]:
            m = (l + r) // 2
            nxt += [(l, m), (m, r)]
        levels.append(nxt)
    return levels

for lv in halving_tree(12):
    print(lv)

[(0, 12)]
[(0, 6), (6, 12)]
[(0, 3), (3, 6), (6, 9), (9, 12)]


### 3.4. `sys.setrecursionlimit` và độ sâu đệ quy

Độ sâu đệ quy ở đây chỉ là số tầng (≤ 17 với `n = 10^5`), nên đệ quy an toàn. (Khác với đệ quy trên danh sách liên kết hoặc cây lệch.)

In [5]:
print("giới hạn đệ quy:", sys.getrecursionlimit(), "| số tầng tối đa với n = 2^16·k:", len(halving_tree(2**16)))

giới hạn đệ quy: 1000 | số tầng tối đa với n = 2^16·k: 17


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi đoạn có độ dài chẵn đứng trước **hai lựa chọn**: giữ nguyên (trả chi phí của nó) hoặc tách đôi (trả chi phí tốt nhất của hai nửa). Hai nửa **độc lập** với nhau ⇒ cấu trúc con tối ưu.

Tính tay ví dụ 1 (`"1010"`, `enc = 2`, `flat = 1`):

| đoạn | giữ nguyên | tách | tốt nhất |
|---|---|---|---|
| "1" | 1·1·2 = 2 | – | 2 |
| "0" | flat = 1 | – | 1 |
| "10" | 2·1·2 = 4 | 2 + 1 = 3 | 3 |
| "1010" | 4·2·2 = 16 | 3 + 3 = 6 | **6** |

Trực giác: chi phí `L · X · enc` tăng gần như **bình phương** khi gộp, nên tách thường có lợi; nhưng đoạn toàn `0` tách ra thì phải trả `flatCost` cho **mỗi** mảnh — khi `flatCost` lớn thì giữ nguyên lại tốt hơn (ví dụ 2).

## 5. Cách 1 — Brute Force: liệt kê mọi cách phân hoạch hợp lệ

Mỗi phân hoạch hợp lệ là một "tỉa" của cây chia đôi. Đệ quy trả về **danh sách mọi chi phí có thể** của một đoạn rồi lấy min ở gốc. Số phân hoạch tăng nhanh (cỡ hàm mũ), chỉ dùng cho `n` nhỏ.

In [6]:
def all_costs(s, l, r, enc, flat):
    own = seg_cost(s[l:r], enc, flat)
    res = [own]
    if (r - l) % 2 == 0:
        m = (l + r) // 2
        left = all_costs(s, l, m, enc, flat)
        right = all_costs(s, m, r, enc, flat)
        res += [a + b for a in left for b in right]
    return res

def brute(s, enc, flat):
    return min(all_costs(s, 0, len(s), enc, flat))

for s_, e, f, exp in examples:
    print(brute(s_, e, f), "kỳ vọng", exp)
print("số phân hoạch của xâu độ dài 16:", len(all_costs("1" * 16, 0, 16, 1, 1)))

6 kỳ vọng 6
12 kỳ vọng 12
2 kỳ vọng 2
số phân hoạch của xâu độ dài 16: 677


**Nút thắt:** liệt kê toàn bộ tổ hợp. Nhưng ta chỉ cần **giá trị nhỏ nhất** của mỗi đoạn — một số duy nhất ⇒ quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

**Công thức:**

`best(l, r) = cost(l, r)` nếu `r - l` lẻ;

`best(l, r) = min(cost(l, r), best(l, m) + best(m, r))` nếu `r - l` chẵn, với `m = (l + r) / 2`.

**Đột phá 1:** vì cách tách là **duy nhất** (ở giữa), mỗi đoạn chỉ xuất hiện một lần trong cây ⇒ không cần memo; số lần gọi ≤ `2n`.

**Đột phá 2:** dùng tiền tố, `cost(l, r)` là `O(1)` ⇒ tổng `O(n)`.

**Quy tắc "nếu … thì …":** nếu đoạn lẻ ⇒ bắt buộc giữ nguyên; nếu chẵn ⇒ so sánh hai phương án.

**Phân tích một bước tách (để hiểu khi nào nên tách).** Xét đoạn độ dài `L` có `X > 0` ký tự `1`, chia thành hai nửa có `x₁ + x₂ = X` (chỉ xét trường hợp **không tách tiếp** hai nửa):

- Nếu **cả hai** nửa đều có `1`: chi phí tách `= (L/2)·x₁·enc + (L/2)·x₂·enc = L·X·enc / 2` — luôn bằng **một nửa** chi phí giữ nguyên ⇒ tách luôn lợi.
- Nếu một nửa **toàn 0**: chi phí tách `= (L/2)·X·enc + flat`. Tách lợi ⇔ `flat < L·X·enc / 2`.
- Nếu `X = 0`: giữ nguyên `flat`, tách ít nhất `2·flat` ⇒ **không bao giờ** tách đoạn toàn 0.

Vì sau khi tách còn có thể tách tiếp, quyết định đúng phải so sánh với `best` của hai nửa — nhưng phân tích trên giải thích vì sao đáp án thường "tách quanh các ký tự `1`, giữ nguyên các khối `0` lớn".

In [7]:
# Kiểm chứng phân tích: đoạn toàn 0 không bao giờ nên tách
rng = random.Random(0)
ok = True
for _ in range(300):
    L = 2 ** rng.randint(1, 4); e, f = rng.randint(1, 50), rng.randint(1, 50)
    ok &= brute("0" * L, e, f) == f
print("đoạn toàn 0 luôn giữ nguyên:", ok)
for L, X, e, f in [(8, 1, 1, 3), (8, 1, 1, 5), (8, 2, 1, 100)]:
    s_ = "1" * X + "0" * (L - X)
    print(f"L={L}, X={X}, enc={e}, flat={f}: giữ={L * X * e}, tách một lần={L // 2 * X * e + f} -> tốt nhất={brute(s_, e, f)}")

đoạn toàn 0 luôn giữ nguyên: True
L=8, X=1, enc=1, flat=3: giữ=8, tách một lần=7 -> tốt nhất=7
L=8, X=1, enc=1, flat=5: giữ=8, tách một lần=9 -> tốt nhất=8
L=8, X=2, enc=1, flat=100: giữ=16, tách một lần=108 -> tốt nhất=16


## 7. Các cách tiếp theo

- **Cách 2:** đệ quy + `s[l:r].count('1')` — đúng, `O(n log n)` (mỗi tầng đếm tổng `n` ký tự).
- **Cách 3:** đệ quy + **tổng tiền tố** — `O(n)` (lời giải nộp chính).
- **Cách 4:** **DP từ dưới lên theo tầng** (không đệ quy) — `O(n)`.

### 7.1. Cách 2 — Đệ quy với cắt xâu

In [8]:
def rec_slice(s, enc, flat):
    def best(l, r):
        res = seg_cost(s[l:r], enc, flat)
        if (r - l) % 2 == 0:
            m = (l + r) // 2
            res = min(res, best(l, m) + best(m, r))
        return res
    return best(0, len(s))

for s_, e, f, exp in examples:
    print(rec_slice(s_, e, f), "kỳ vọng", exp)

6 kỳ vọng 6
12 kỳ vọng 12
2 kỳ vọng 2


**Độ phức tạp:** mỗi tầng cắt và đếm tổng cộng `n` ký tự, có `O(log n)` tầng ⇒ `O(n log n)`; bộ nhớ tạm `O(n)` cho các lát cắt.

### 7.2. Cách 3 — Đệ quy + tổng tiền tố (lời giải chính)

**Giải thích từng đoạn:**

1. `pre[i]` = số `'1'` trong `s[0..i-1]`.
2. `best(l, r)`: `x = pre[r] - pre[l]`; `res = (r - l)·x·enc` nếu `x > 0`, ngược lại `flat`.
3. Nếu `(r - l)` chẵn: `res = min(res, best(l, m) + best(m, r))`.

In [9]:
def rec_prefix(s, enc, flat):
    n = len(s)
    pre = [0] * (n + 1)
    for i, c in enumerate(s):
        pre[i + 1] = pre[i] + (c == '1')

    def best(l, r):
        x = pre[r] - pre[l]
        res = (r - l) * x * enc if x else flat
        if (r - l) % 2 == 0:
            m = (l + r) // 2
            res = min(res, best(l, m) + best(m, r))
        return res

    return best(0, n)

for s_, e, f, exp in examples:
    print(rec_prefix(s_, e, f), "kỳ vọng", exp)

6 kỳ vọng 6
12 kỳ vọng 12
2 kỳ vọng 2


**Độ phức tạp Cách 3:** số nút cây ≤ `2n`, mỗi nút `O(1)` ⇒ `O(n)` thời gian, `O(n)` bộ nhớ (mảng tiền tố) + `O(log n)` ngăn xếp đệ quy.

### 7.3. Cách 4 — DP từ dưới lên theo tầng

**Ý tưởng:** độ dài đoạn ở tầng `t` là `n / 2^t`. Tầng sâu nhất có độ dài **lẻ** (lá). Tính `best` cho mọi đoạn của tầng sâu nhất (chỉ là chi phí giữ nguyên), rồi đi lên: `best_t[j] = min(cost của đoạn j ở tầng t, best_{t+1}[2j] + best_{t+1}[2j+1])`.

In [10]:
def bottom_up(s, enc, flat):
    n = len(s)
    pre = [0] * (n + 1)
    for i, c in enumerate(s):
        pre[i + 1] = pre[i] + (c == '1')

    def cost(l, r):
        x = pre[r] - pre[l]
        return (r - l) * x * enc if x else flat

    L = n
    lengths = [L]
    while L % 2 == 0:
        L //= 2
        lengths.append(L)
    # tầng sâu nhất
    L = lengths[-1]
    best = [cost(l, l + L) for l in range(0, n, L)]
    for L in reversed(lengths[:-1]):
        best = [min(cost(j * L, (j + 1) * L), best[2 * j] + best[2 * j + 1]) for j in range(n // L)]
    return best[0]

for s_, e, f, exp in examples:
    print(bottom_up(s_, e, f), "kỳ vọng", exp)

6 kỳ vọng 6
12 kỳ vọng 12
2 kỳ vọng 2


In [11]:
class SolutionBottomUp:
    def minCost(self, s: str, encCost: int, flatCost: int) -> int:
        return bottom_up(s, encCost, flatCost)

print([SolutionBottomUp().minCost(s_, e, f) for s_, e, f, _ in examples])

[6, 12, 2]


**Độ phức tạp Cách 4:** tổng số đoạn trên mọi tầng `n/L_min + … + 1 ≤ 2n` ⇒ `O(n)`; không đệ quy.

## 8. TRACE từng bước

Trace Cách 3 trên `s = "11000000"`, `enc = 1`, `flat = 3` (tự dựng).

| đoạn | X | giữ nguyên | tách | best |
|---|---|---|---|---|
| "1" ×2 | 1 | 1 | – | 1 |
| "0" ×6 | 0 | 3 | – | 3 |
| "11" | 2 | 2·2·1 = 4 | 1 + 1 = 2 | 2 |
| "00" ×3 | 0 | 3 | 3 + 3 = 6 | 3 |
| "1100" | 2 | 8 | 2 + 3 = 5 | 5 |
| "0000" | 0 | 3 | 3 + 3 = 6 | 3 |
| "11000000" | 2 | 16 | 5 + 3 = 8 | **8** |

Đoạn toàn `0` nên **giữ nguyên** (một lần `flat` rẻ hơn nhiều lần), đoạn có `1` nên tách nhỏ quanh các ký tự `1`.

In [12]:
def trace(s, enc, flat):
    n = len(s); pre = [0] * (n + 1)
    for i, c in enumerate(s): pre[i + 1] = pre[i] + (c == '1')
    def best(l, r, depth=0):
        x = pre[r] - pre[l]
        own = (r - l) * x * enc if x else flat
        res, how = own, "giữ"
        if (r - l) % 2 == 0:
            m = (l + r) // 2
            sp = best(l, m, depth + 1) + best(m, r, depth + 1)
            if sp < own:
                res, how = sp, f"tách ({sp})"
        print("  " * depth + f"[{l},{r}) {s[l:r]!r} X={x} giữ={own} -> best={res} ({how})")
        return res
    return best(0, n)

print("kết quả:", trace("11000000", 1, 3))

      [0,1) '1' X=1 giữ=1 -> best=1 (giữ)
      [1,2) '1' X=1 giữ=1 -> best=1 (giữ)
    [0,2) '11' X=2 giữ=4 -> best=2 (tách (2))
      [2,3) '0' X=0 giữ=3 -> best=3 (giữ)
      [3,4) '0' X=0 giữ=3 -> best=3 (giữ)
    [2,4) '00' X=0 giữ=3 -> best=3 (giữ)
  [0,4) '1100' X=2 giữ=8 -> best=5 (tách (5))
      [4,5) '0' X=0 giữ=3 -> best=3 (giữ)
      [5,6) '0' X=0 giữ=3 -> best=3 (giữ)
    [4,6) '00' X=0 giữ=3 -> best=3 (giữ)
      [6,7) '0' X=0 giữ=3 -> best=3 (giữ)
      [7,8) '0' X=0 giữ=3 -> best=3 (giữ)
    [6,8) '00' X=0 giữ=3 -> best=3 (giữ)
  [4,8) '0000' X=0 giữ=3 -> best=3 (giữ)
[0,8) '11000000' X=2 giữ=16 -> best=8 (tách (8))
kết quả: 8


In [13]:
# Trace Cách 4 (theo tầng) trên ví dụ 1
s_, e, f = "1010", 2, 1
n = len(s_); pre = [0] * (n + 1)
for i, c in enumerate(s_): pre[i + 1] = pre[i] + (c == '1')
cost = lambda l, r: (r - l) * (pre[r] - pre[l]) * e if pre[r] - pre[l] else f
best = [cost(i, i + 1) for i in range(n)]
print("tầng độ dài 1:", best)
L = 2
while L <= n:
    best = [min(cost(j * L, (j + 1) * L), best[2 * j] + best[2 * j + 1]) for j in range(n // L)]
    print(f"tầng độ dài {L}:", best)
    L *= 2

tầng độ dài 1: [2, 1, 2, 1]
tầng độ dài 2: [3, 3]
tầng độ dài 4: [6]


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minCost(self, s: str, encCost: int, flatCost: int) -> int:
        n = len(s)
        pre = [0] * (n + 1)
        for i, c in enumerate(s):
            pre[i + 1] = pre[i] + (c == '1')

        def best(l, r):
            x = pre[r] - pre[l]
            res = (r - l) * x * encCost if x else flatCost
            if (r - l) % 2 == 0:
                m = (l + r) // 2
                split = best(l, m) + best(m, r)
                if split < res:
                    res = split
            return res

        return best(0, n)

sol = Solution()
for s_, e, f, exp in examples:
    got = sol.minCost(s_, e, f)
    print(repr(s_), "->", got, "✔" if got == exp else "✘")

'1010' -> 6 ✔
'1010' -> 12 ✔
'00' -> 2 ✔


## 10. Vì sao thuật toán đúng

1. **Không gian lời giải:** mọi phân hoạch hợp lệ sinh ra từ đoạn `[0, n)` bằng cách quyết định, tại mỗi đoạn chẵn đang có, "tách" hay "không tách". Vì cách tách là duy nhất, mỗi phân hoạch tương ứng một **cây con gốc** của cây chia đôi.
2. **Cấu trúc con tối ưu:** nếu đoạn `[l, r)` bị tách, chi phí là tổng chi phí của hai nửa, và quyết định trong hai nửa độc lập ⇒ tối ưu toàn cục cần tối ưu từng nửa. Nếu không tách, chi phí cố định `cost(l, r)`. Vậy `best(l, r)` = min hai trường hợp (quy nạp theo độ dài).
3. **Đúng chi phí:** `pre[r] - pre[l]` là số `'1'` trong đoạn; công thức chi phí áp dụng đúng định nghĩa.
4. **Dừng:** độ dài giảm một nửa mỗi lần; đoạn lẻ là lá.

## 11. Phân tích Time / Space Complexity

Đặt `n = |s|`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê phân hoạch | hàm mũ theo số nút | lớn |
| 2. Đệ quy + cắt xâu | `O(n log n)` | `O(n)` |
| 3. Đệ quy + tiền tố | `O(n)` | `O(n)` |
| 4. Từ dưới lên | `O(n)` | `O(n)` |

Giá trị lớn nhất có thể: `n · n · enc = 10^{10} · 10^5 = 10^{15}` — Python không lo tràn, các ngôn ngữ khác cần `long long`.

**Đếm số nút chính xác.** Viết `n = 2^t · q` với `q` lẻ. Cây chia đôi có `t + 1` tầng; tầng thứ `i` (từ gốc, `i = 0..t`) có `2^i` đoạn độ dài `n / 2^i`. Tổng số nút `= 2^{t+1} - 1 ≤ 2n - 1` (vì `2^t ≤ n`). Khi `n` lẻ (`t = 0`) chỉ có **một** nút: không tách được gì, đáp án là chi phí cả xâu. Khi `n` là lũy thừa của 2, cây đầy đủ tới từng ký tự.

In [15]:
for n in (1, 7, 12, 96, 1024, 99999, 98304):
    t = (n & -n).bit_length() - 1
    print(f"n={n}: t={t}, số tầng={t + 1}, số nút={2 ** (t + 1) - 1}")

n=1: t=0, số tầng=1, số nút=1
n=7: t=0, số tầng=1, số nút=1
n=12: t=2, số tầng=3, số nút=7
n=96: t=5, số tầng=6, số nút=63
n=1024: t=10, số tầng=11, số nút=2047
n=99999: t=0, số tầng=1, số nút=1
n=98304: t=15, số tầng=16, số nút=65535


## 12. Bộ test

In [16]:
def run_all(s, e, f, with_brute=True):
    res = [rec_slice(s, e, f), rec_prefix(s, e, f), bottom_up(s, e, f), Solution().minCost(s, e, f)]
    if with_brute:
        res.append(brute(s, e, f))
    return res

for s_, e, f, exp in examples:
    assert all(v == exp for v in run_all(s_, e, f))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [17]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng liệt kê)
cases = [
    ("1", 5, 7, 5),              # 1 ký tự '1'
    ("0", 5, 7, 7),              # 1 ký tự '0'
    ("111", 2, 1, 18),           # độ dài lẻ: không tách được, 3·3·2
    ("0000", 1, 100, 100),       # toàn 0, flat lớn: giữ nguyên
    ("1111", 1, 1, 4),           # tách hết: 4 đoạn "1"
    ("100000", 3, 1, None),      # 6 -> 3 + 3 (lẻ)
]
for s_, e, f, exp in cases:
    r = run_all(s_, e, f)
    assert len(set(r)) == 1, (s_, r)
    if exp is not None:
        assert r[0] == exp, (s_, r)
    print(repr(s_), e, f, "->", r[0])

'1' 5 7 -> 5
'0' 5 7 -> 7
'111' 2 1 -> 18
'0000' 1 100 -> 100
'1111' 1 1 -> 4
'100000' 3 1 -> 10


In [18]:
# 12.3 phản ví dụ cho tham lam "luôn tách khi có thể"
def always_split(s, e, f):
    def best(l, r):
        if (r - l) % 2 == 0:
            m = (l + r) // 2
            return best(l, m) + best(m, r)
        x = s[l:r].count('1')
        return (r - l) * x * e if x else f
    return best(0, len(s))

print("luôn tách:", always_split("1010", 3, 10), "| đúng:", Solution().minCost("1010", 3, 10))
# và "không bao giờ tách"
print("không tách:", seg_cost("1010", 2, 1), "| đúng:", Solution().minCost("1010", 2, 1))

luôn tách: 26 | đúng: 12
không tách: 16 | đúng: 6


In [19]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3864)
for _ in range(500):
    n = rng.choice([1, 2, 3, 4, 5, 6, 8, 12, 16])
    s_ = "".join(rng.choice("01") for _ in range(n))
    e, f = rng.randint(1, 10), rng.randint(1, 30)
    r = run_all(s_, e, f)
    assert len(set(r)) == 1, (s_, e, f, r)
print("500 test nhỏ (có liệt kê): OK")

500 test nhỏ (có liệt kê): OK


In [20]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
for n in (10**5, 2**16, 99999, 3 * 2**15):
    s_ = "".join(rng.choice("0001") for _ in range(n))
    t0 = time.time(); a = Solution().minCost(s_, rng.randint(1, 10**5), rng.randint(1, 10**5)); t1 = time.time()
    print(f"n={n}: thời gian {t1 - t0:.3f}s")
s_ = "".join(rng.choice("01") for _ in range(2**16))
print("khớp 3 cách trên n = 2^16:", len({rec_slice(s_, 7, 9), rec_prefix(s_, 7, 9), bottom_up(s_, 7, 9)}) == 1)

n=100000: thời gian 0.007s


n=65536: thời gian 0.022s


n=99999: thời gian 0.008s


n=98304: thời gian 0.017s


khớp 3 cách trên n = 2^16: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luôn tách khi độ dài chẵn

Đã thấy ở 12.3: với `flatCost` lớn, tách đoạn toàn `0` hoặc tách quá nhỏ làm tăng chi phí. **Sửa:** luôn so sánh với phương án giữ nguyên.

### Lỗi 2 — Dùng `L · X · enc` cho cả đoạn toàn `0` (ra 0)

In [21]:
def wrong_flat(s, e, f):
    n = len(s); pre = [0] * (n + 1)
    for i, c in enumerate(s): pre[i + 1] = pre[i] + (c == '1')
    def best(l, r):
        res = (r - l) * (pre[r] - pre[l]) * e            # SAI: X = 0 cho chi phí 0, phải là flatCost
        if (r - l) % 2 == 0:
            m = (l + r) // 2; res = min(res, best(l, m) + best(m, r))
        return res
    return best(0, n)

print("sai:", wrong_flat("00", 1, 2), "| đúng:", Solution().minCost("00", 1, 2))

sai: 0 | đúng: 2


### Lỗi 3 — Tách cả đoạn có độ dài lẻ (lệch một)

Nếu viết `m = (l + r) // 2` và tách mọi đoạn có độ dài `> 1`, các nửa sẽ lệch (`⌊L/2⌋`, `⌈L/2⌉`) — vi phạm đề.

In [22]:
def wrong_split_odd(s, e, f):
    def best(l, r):
        x = s[l:r].count('1'); res = (r - l) * x * e if x else f
        if r - l > 1:                                      # SAI: phải là độ dài chẵn
            m = (l + r) // 2; res = min(res, best(l, m) + best(m, r))
        return res
    return best(0, len(s))

print("sai:", wrong_split_odd("111", 2, 1), "| đúng:", Solution().minCost("111", 2, 1))

sai: 6 | đúng: 18


### Lỗi 4 — Memo bằng `@cache` trên `(l, r)` rồi tưởng là cần thiết

Không sai, nhưng **thừa**: mỗi đoạn chỉ được gọi một lần (cây chia đôi không có nút chung). Memo chỉ tốn bộ nhớ. Hiểu rõ cấu trúc giúp tránh tối ưu nhầm chỗ.

In [23]:
calls = 0
def count_calls(l, r):
    global calls
    calls += 1
    if (r - l) % 2 == 0:
        m = (l + r) // 2; count_calls(l, m); count_calls(m, r)
count_calls(0, 2**10)
print("số lần gọi với n = 1024:", calls, "= 2n - 1 =", 2 * 1024 - 1)

số lần gọi với n = 1024: 2047 = 2n - 1 = 2047


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chỉ được tách đôi ở giữa"** ⇒ cây chia đôi cố định, đệ quy `min(giữ, tách)` — không cần memo.
- **Chi phí đoạn phụ thuộc số lượng phần tử đặc biệt** ⇒ tổng tiền tố cho `O(1)`.
- Hai lực ngược chiều (`L · X` muốn tách nhỏ, `flatCost` muốn gộp) ⇒ không có tham lam đơn giản, phải so sánh ở mọi nút.
- Liên quan: *Minimum Cost to Make Array Equal* (khác), các bài "segment tree-like DP" trên cây nhị phân hoàn chỉnh; LC 3523/3326 kiểu chia đôi theo lũy thừa 2.

## 15. Biến thể của bài

1. **Được tách ở vị trí bất kỳ** (không chỉ giữa): DP đoạn `O(n³)` hoặc DP tiền tố `O(n²)` (vì tổng chi phí cộng tính, tương đương chia xâu thành các khúc liên tiếp).
2. **Tách thành `k` phần bằng nhau** khi độ dài chia hết cho `k`: cây `k`-phân.
3. **Có thêm chi phí mỗi lần tách** `splitCost`: cộng vào nhánh tách.
4. **Trả về số đoạn** của phân hoạch tối ưu (hoặc chính phân hoạch).

In [24]:
# Biến thể 1: tách ở vị trí bất kỳ -> DP tiền tố dp[i] = min_j dp[j] + cost(j, i)
def any_split(s, e, f):
    n = len(s); pre = [0] * (n + 1)
    for i, c in enumerate(s): pre[i + 1] = pre[i] + (c == '1')
    cost = lambda l, r: (r - l) * (pre[r] - pre[l]) * e if pre[r] - pre[l] else f
    dp = [0] + [float('inf')] * n
    for i in range(1, n + 1):
        dp[i] = min(dp[j] + cost(j, i) for j in range(i))
    return dp[n]

for s_, e, f, _ in examples + [("111", 2, 1, None), ("100000", 3, 1, None)]:
    print(repr(s_), "chia đôi:", Solution().minCost(s_, e, f), "| tách tự do:", any_split(s_, e, f))

'1010' chia đôi: 6 | tách tự do: 6
'1010' chia đôi: 12 | tách tự do: 12
'00' chia đôi: 2 | tách tự do: 2
'111' chia đôi: 18 | tách tự do: 6
'100000' chia đôi: 10 | tách tự do: 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | Mọi phân hoạch | mũ | lớn | Kiểm thử |
| 2. Đệ quy + cắt xâu | `min(giữ, tách)` | `O(n log n)` | `O(n)` | Code ngắn nhất |
| 3. Đệ quy + tiền tố | `O(1)` mỗi nút | `O(n)` | `O(n)` | **Lời giải chuẩn** |
| 4. Từ dưới lên | Theo tầng | `O(n)` | `O(n)` | Tránh đệ quy |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minCost("11", 5, 1)` = ?

**Bài 2 (tính tay).** `minCost("10000001", 1, 4)` — lập bảng theo tầng.

**Bài 3 (điền TODO).** Viết `min_cost_with_split_fee(s, enc, flat, fee)`: mỗi lần tách tốn thêm `fee`.

**Bài 4 (tự viết lại).** Viết lại Cách 3 trả về thêm **danh sách đoạn** của phân hoạch tối ưu.

**Bài 5 (mini challenge).** Biến thể `k`-phân: đoạn có độ dài chia hết cho `k` được tách thành `k` phần bằng nhau. Viết lời giải và kiểm chứng bằng liệt kê.

In [25]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_cost_with_split_fee(s, enc, flat, fee):
    # TODO: như Cách 3 nhưng nhánh tách cộng thêm fee
    pass

In [26]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_cost_k_ary(s, enc, flat, k):
    # TODO: nếu (r - l) % k == 0 và r - l > 1 thì thử tách thành k đoạn bằng nhau
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Giữ nguyên: `2·2·5 = 20`. (2) Tách: `5 + 5 = 10`. (3) Đáp án `10`.

**Bài 2.** (1) Tầng 1 ký tự: `1` hoặc `4`. (2) Tầng 2: `"10"` → min(2, 1+4=5) = 2; `"00"` → min(4, 8) = 4. (3) Tầng 4: `"1000"` → min(4, 2+4=6) = 4; tương tự `"0001"` → 4; gốc min(16, 8) = `8`.

**Bài 3.** (1) `split = best(l,m) + best(m,r) + fee`. (2) `fee` lớn ⇒ ít tách. (3) `fee = 0` trùng bài gốc.

**Bài 4.** (1) Hàm trả về `(chi phí, danh sách)`. (2) Khi tách thì nối hai danh sách. (3) Kiểm tra tổng chi phí các đoạn bằng chi phí trả về.

**Bài 5.** (1) Điều kiện tách: độ dài chia hết cho `k` và `> 1`. (2) Tổng `best` của `k` phần. (3) Liệt kê như Cách 1 để đối chiếu.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().minCost("11", 5, 1) == 10
assert Solution().minCost("10000001", 1, 4) == 8 == brute("10000001", 1, 4)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [28]:
# Bài 3
def min_cost_with_split_fee_sol(s, enc, flat, fee):
    n = len(s); pre = [0] * (n + 1)
    for i, c in enumerate(s): pre[i + 1] = pre[i] + (c == '1')
    def best(l, r):
        x = pre[r] - pre[l]; res = (r - l) * x * enc if x else flat
        if (r - l) % 2 == 0:
            m = (l + r) // 2; res = min(res, best(l, m) + best(m, r) + fee)
        return res
    return best(0, n)

assert min_cost_with_split_fee_sol("1010", 2, 1, 0) == 6
assert min_cost_with_split_fee_sol("1010", 2, 1, 100) == 16
print("Bài 3 OK:", [min_cost_with_split_fee_sol("1010", 2, 1, fee) for fee in (0, 1, 3, 5, 100)])

Bài 3 OK: [6, 9, 11, 13, 16]


In [29]:
# Bài 4
def min_cost_with_parts(s, enc, flat):
    n = len(s); pre = [0] * (n + 1)
    for i, c in enumerate(s): pre[i + 1] = pre[i] + (c == '1')
    def best(l, r):
        x = pre[r] - pre[l]; own = (r - l) * x * enc if x else flat
        res = (own, [s[l:r]])
        if (r - l) % 2 == 0:
            m = (l + r) // 2; a, b = best(l, m), best(m, r)
            if a[0] + b[0] < own: res = (a[0] + b[0], a[1] + b[1])
        return res
    return best(0, n)

c, parts = min_cost_with_parts("11000000", 1, 3)
assert c == Solution().minCost("11000000", 1, 3) == sum(seg_cost(p, 1, 3) for p in parts)
print("Bài 4 OK:", c, parts)

Bài 4 OK: 8 ['1', '1', '00', '0000']


In [30]:
# Bài 5
def min_cost_k_ary_sol(s, enc, flat, k):
    n = len(s); pre = [0] * (n + 1)
    for i, c in enumerate(s): pre[i + 1] = pre[i] + (c == '1')
    def best(l, r):
        x = pre[r] - pre[l]; res = (r - l) * x * enc if x else flat
        L = r - l
        if L > 1 and L % k == 0:
            step = L // k
            res = min(res, sum(best(l + i * step, l + (i + 1) * step) for i in range(k)))
        return res
    return best(0, n)

def brute_k(s, enc, flat, k):
    def costs(l, r):
        own = [seg_cost(s[l:r], enc, flat)]
        L = r - l
        if L > 1 and L % k == 0:
            step = L // k; combos = [0]
            for i in range(k):
                sub = costs(l + i * step, l + (i + 1) * step)
                combos = [a + b for a in combos for b in sub]
            own += combos
        return own
    return min(costs(0, len(s)))

rng = random.Random(5)
for _ in range(200):
    k = rng.choice([2, 3]); n = rng.choice([1, 2, 3, 4, 6, 9, 8])
    s_ = "".join(rng.choice("01") for _ in range(n)); e, f = rng.randint(1, 9), rng.randint(1, 20)
    assert min_cost_k_ary_sol(s_, e, f, k) == brute_k(s_, e, f, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pre[i+1] = pre[i] + (s[i] == '1')
def best(l, r):
    x = pre[r] - pre[l]
    res = (r - l) * x * enc if x else flat
    if (r - l) % 2 == 0:
        m = (l + r) // 2
        res = min(res, best(l, m) + best(m, r))
    return res
return best(0, n)
```

**Checklist:**

- [ ] Đoạn toàn `0` trả `flatCost` (không phải 0)?
- [ ] Chỉ tách khi độ dài **chẵn**, tách đúng giữa?
- [ ] Luôn so sánh với phương án **không tách**?
- [ ] Đếm số `'1'` bằng tiền tố (không cắt xâu nhiều lần)?
- [ ] Đáp án có thể tới `10^{15}` (ngôn ngữ khác cần kiểu 64-bit)?